What’s going on

•	The .mtx is a Matrix Market sparse matrix. In this GEO series, it’s genes × cells (not cells × genes). The series file also confirms this: “rows are genes, cells are columns” for the gene-by-cell count matrix.  ￼

•	Your genes CSV and barcodes CSV do not have headers. When you read them with default header=0, Pandas consumes the first row as a header, causing the length mismatch (22468 vs 22469).



From the files you uploaded, I just verified:
•	barcodes: shape (100955, 1) (no header)

•	genes: shape (22469, 1) (no header)

•	lookup (barcode → cell.type, tissue.type): shape (100955, 3) with columns barcode, cell.type, tissue.type


## A) Build an AnnData, then aggregate to neuron classes

In [ ]:
# === GSE136049 → neuron-class transcriptomes, with gene symbol mapping ===
# Requirements: scanpy, scipy, pandas, numpy
# - Assumes the following files are in your working directory (rename paths if needed):
#   * GSE136049_gene_by_barcode_count_matrix_all_cells.mtx
#   * GSE136049_all_cells_gene_annotations.csv
#   * GSE136049_all_cells_barcodes_column_names.csv
#   * GSE136049_cell_type_annotation_lookup_table.csv
#   * GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf.gz  (for ID→symbol map)
#
# Outputs:
#   * GSE136049_gene_by_neuronclass_counts.csv
#   * GSE136049_gene_by_neuronclass_lognorm_means.csv
#   * (or transposed versions, if ORIENTATION = "neurons_by_genes")

import gzip
import re
import numpy as np
import pandas as pd
from scipy.io import mmread
from scipy import sparse
import scanpy as sc

# ---------------------------
# Config (change these paths)
# ---------------------------
# Data downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE136049
MTX_PATH      = "GSE136049/GSE136049_gene_by_barcode_count_matrix_all_cells.mtx"
GENES_PATH    = "GSE136049/GSE136049_all_cells_gene_annotations.csv"           # one WBGene per line, no header
BARCODES_PATH = "GSE136049/GSE136049_all_cells_barcodes_column_names.csv"      # one barcode per line, no header
LOOKUP_PATH   = "GSE136049/GSE136049_cell_type_annotation_lookup_table.csv"    # columns: barcode,cell.type,tissue.type
GTF_PATH      = "GSE136049/GSE136049_c_elegans.PRJNA13758.WS273.canonical_geneset.extend3UTR_optimized.gtf.gz"

# Keep only neurons? Set to False to keep all tissues.
KEEP_ONLY_NEURONS = True
# Drop cells whose cell.type == "Unannotated"
DROP_UNANNOTATED = True
# Output orientation: "genes_by_neurons" (default) or "neurons_by_genes"
ORIENTATION = "genes_by_neurons"
# Normalization target sum (CP10K before log1p)
TARGET_SUM = 1e4

# ---------------------------
# Helpers
# ---------------------------

def parse_gtf_gene_name_map(gtf_path):
    """
    Parse a (possibly gzipped) GTF to build a map: WormBase gene_id -> gene_name.
    Only 'gene' feature lines are used; if some names are missing we’ll fill later with the ID.

    Returns:
        dict: { "WBGene00000001": "gene-1", ... }
    """
    open_fn = gzip.open if gtf_path.endswith(".gz") else open
    gene_id_to_name = {}
    attr_re = re.compile(r'(\S+)\s+"([^"]+)"')
    with open_fn(gtf_path, "rt") as fh:
        for line in fh:
            if not line or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9:
                continue
            feature = fields[2]
            if feature != "gene":
                continue
            attrs = fields[8]
            # Parse attributes; find gene_id and gene_name
            parsed = dict(attr_re.findall(attrs))
            gid = parsed.get("gene_id") or parsed.get("geneID") or parsed.get("geneId")
            gname = parsed.get("gene_name") or parsed.get("Name") or parsed.get("gene")
            if gid:
                if gname:
                    gene_id_to_name[gid] = gname
                else:
                    # If no gene_name provided, leave missing (we’ll backfill with gid)
                    gene_id_to_name.setdefault(gid, None)
    return gene_id_to_name

def aggregate_duplicates(df, how="sum"):
    """
    Aggregate rows with duplicate indices (gene symbols) by sum or mean.
    """
    if how == "sum":
        return df.groupby(df.index).sum()
    elif how == "mean":
        return df.groupby(df.index).mean()
    else:
        raise ValueError("how must be 'sum' or 'mean'")

def ensure_orientation(df, orientation="genes_by_neurons"):
    if orientation == "genes_by_neurons":
        return df
    elif orientation == "neurons_by_genes":
        return df.T
    else:
        raise ValueError("ORIENTATION must be 'genes_by_neurons' or 'neurons_by_genes'")

# ---------------------------
# Load data
# ---------------------------

print("Reading matrix market (genes x cells)...")
X = mmread(MTX_PATH).tocsr()  # shape: (n_genes, n_cells)

print("Reading gene IDs and barcodes...")
genes = pd.read_csv(GENES_PATH, header=None)[0].astype(str).values
barcodes = pd.read_csv(BARCODES_PATH, header=None)[0].astype(str).values
lookup = pd.read_csv(LOOKUP_PATH)

assert X.shape[0] == len(genes), f"Gene count mismatch: {X.shape[0]} vs {len(genes)}"
assert X.shape[1] == len(barcodes), f"Cell count mismatch: {X.shape[1]} vs {len(barcodes)}"

# Create AnnData as cells x genes
print("Constructing AnnData (cells x genes)...")
adata = sc.AnnData(X=X.T)  # (cells, genes)
adata.obs_names = barcodes
adata.var_names = genes
adata.var_names_make_unique()

# Attach cell-type/tissue annotations
lookup = lookup.set_index("barcode").reindex(adata.obs_names)
adata.obs["cell.type"] = lookup["cell.type"].astype("category")
adata.obs["tissue.type"] = lookup["tissue.type"].astype("category")

# Optionally restrict to neurons and drop unannotated
mask = np.ones(adata.n_obs, dtype=bool)
if KEEP_ONLY_NEURONS:
    mask &= (adata.obs["tissue.type"].astype(str).values == "Neuron")
if DROP_UNANNOTATED:
    mask &= (adata.obs["cell.type"].astype(str).values != "Unannotated")

adata = adata[mask].copy()
adata.obs["cell.type"] = adata.obs["cell.type"].astype("category")  # compact categories

# ---------------------------
# Build cell-type indicator and aggregate (fast & memory-friendly)
# ---------------------------
print("Aggregating raw UMI counts by neuron class...")
ct = adata.obs["cell.type"].astype("category")
ct_names = list(ct.cat.categories)

rows = np.arange(adata.n_obs)
cols = ct.cat.codes.values
S = sparse.csr_matrix((np.ones_like(rows), (rows, cols)),
                      shape=(adata.n_obs, len(ct_names)))
# genes x types = (genes x cells) @ (cells x types)
counts_by_type = adata.X.T @ S  # shape: (n_genes, n_types)

counts_df = pd.DataFrame(
    data=np.asarray(counts_by_type.todense()),
    index=adata.var_names,  # WBGene IDs for now
    columns=ct_names
)

# ---------------------------
# Normalized (CP10K + log1p) means per neuron class
# ---------------------------
print("Computing log-normalized means by neuron class...")
adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm, target_sum=TARGET_SUM)
sc.pp.log1p(adata_norm)

ctn = adata_norm.obs["cell.type"].astype("category")
ctn_names = list(ctn.cat.categories)
rows = np.arange(adata_norm.n_obs)
cols = ctn.cat.codes.values
S_norm = sparse.csr_matrix((np.ones_like(rows), (rows, cols)),
                           shape=(adata_norm.n_obs, len(ctn_names)))
counts_per_type = np.bincount(cols, minlength=len(ctn_names)).astype(float)

means_mat = (adata_norm.X.T @ S_norm).multiply(1.0 / counts_per_type)  # genes x types
means_df = pd.DataFrame(
    data=np.asarray(means_mat.todense()),
    index=adata_norm.var_names,  # WBGene IDs for now
    columns=ctn_names
)

# ---------------------------
# Map WBGene IDs → gene symbols using GTF
# ---------------------------
print("Parsing GTF for WBGene → gene_name map (this can take ~tens of seconds)...")
gid_to_name = parse_gtf_gene_name_map(GTF_PATH)

def apply_symbol_index(df):
    # Map index (WBGene IDs) to symbols; fallback to ID if missing
    wb_ids = df.index.to_series()
    symbols = wb_ids.map(lambda x: gid_to_name.get(x, None) or x)
    df2 = df.copy()
    df2.index = symbols
    # Aggregate duplicates (some IDs map to same symbol)
    df2 = aggregate_duplicates(df2, how=("sum" if df2 is counts_df else "mean"))
    return df2

counts_sym = apply_symbol_index(counts_df)
means_sym  = apply_symbol_index(means_df)

# ---------------------------
# Orientation & save
# ---------------------------
counts_out = ensure_orientation(counts_sym, ORIENTATION)
means_out  = ensure_orientation(means_sym,  ORIENTATION)

# Friendly file names indicating orientation
suffix = "genes_by_neurons" if ORIENTATION == "genes_by_neurons" else "neurons_by_genes"

counts_path = f"GSE136049_{suffix}_counts.csv"
means_path  = f"GSE136049_{suffix}_lognorm_means.csv"

counts_out.to_csv(counts_path)
means_out.to_csv(means_path)

print(f"Done.\nWrote:\n - {counts_path}\n - {means_path}")

ValueError: blocks must be 2-D

Output you’ll get

•	GSE136049_gene_by_neuronclass_counts.csv: raw UMI sums for each gene (rows) in each neuron class (columns).

•	GSE136049_gene_by_neuronclass_lognorm_means.csv: log-normalized mean expression per neuron class (often more useful for “transcriptome profiles”).

If you’d prefer one column per named neuron (e.g., AWA, AVA, …) that’s exactly what the “cell.type” column is giving you—each column in the output corresponds to a neuron class label from the lookup. If you want per-cell instead, just write out the original matrix with barcodes as columns and gene IDs as rows.